# Advanced AI-Driven Search
This notebook demonstrates advanced ai-drive approaches to search. <br>
It specifically covers topics such as learned neural retrieval, mixture of encoders, multi-vector search and wormhole vectors. <br>
The notebook specifically uses Qdrant and SuperLinked to demonstrate these concepts.. <br>

In [1]:
%cd ../

/Users/jakubzovak/Documents/advanced_ai_driven_search


In [2]:
%load_ext autoreload
%autoreload 2

## Setup

In [3]:
import json
import os
from typing import Any, cast, Callable

from datasets import load_dataset
from datasets.dataset_dict import DatasetDict
from datasets.dataset_dict import Dataset
from qdrant_client import QdrantClient
from qdrant_client.models import models
from qdrant_client.http.models.models import QueryResponse
from fastembed import TextEmbedding, SparseTextEmbedding, LateInteractionTextEmbedding
from fastembed.sparse.sparse_embedding_base import SparseEmbedding
from dotenv import load_dotenv

from notebooks.utils import evaluate_retrieval

Load environment variables. **Do not forget to create a .env file in the root directory based on the .env.example file**.

In [5]:
load_dotenv("./.env")

True

Start up local instance of Qdrant through docker.

In [6]:
!docker run -p 6335:6333 -p 6336:6334 -d --name qdrant-server qdrant/qdrant:v1.16

Unable to find image 'qdrant/qdrant:v1.16' locally
v1.16: Pulling from qdrant/qdrant

b700ef54: Pulling fs layer 
03b7ad3b: Pulling fs layer 
b700ef54: Pulling fs layer 
b700ef54: Pulling fs layer 
f04340d3: Pulling fs layer 
b700ef54: Pulling fs layer 
fba1463b: Pulling fs layer 
Digest: sha256:0425e3e03e7fd9b3dc95c4214546afe19de2eb2e28ca621441a56663ac6e1f46
Status: Downloaded newer image for qdrant/qdrant:v1.16
6543ce461d0134f5d7df7089cb4cfe531bfae0d3569bd3c411170de6adb29660


Initiate the Qdrant client by connecting to the server running as a docker container.

In [7]:
client = QdrantClient(host=os.environ["QDRANT_HOST"], port=int(os.environ["QDRANT_PORT"]))

## Dataset

### Task 1 - Data Loading
Load the data from the Hugging Face dataset [Zovi3/pa195_semestral_assignment](https://huggingface.co/datasets/Zovi3/pa195_semestral_assignment/upload/main), explore it and extract/preprocess it if necessary.

In [21]:
query_dataset_dict: DatasetDict = cast(DatasetDict, load_dataset(
    "Zovi3/pa195_semestral_assignment",
    data_dir="query-all-MiniLM-L6-v2-100-filters-embedded-results"
)) 
query_dataset: Dataset = query_dataset_dict["train"]
query_dataset

train.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['text', 'id', 'filters', 'embedding', 'multi_vector_embedding', 'result'],
    num_rows: 100
})

In [8]:
documents_dataset_dict: DatasetDict = cast(DatasetDict, load_dataset(
    "Zovi3/pa195_test",
    name=None, 
    data_dir="corpus-all-MiniLM-L6-v2-50K-groups-multi-vector"
)) 
documents: Dataset = query_dataset_dict["train"]
documents

Using the latest cached version of the dataset since Zovi3/pa195_test couldn't be found on the Hugging Face Hub


ValueError: Couldn't find cache for Zovi3/pa195_test for config 'default-ed30bdc29f56adce'
Available configs in the cache: ['default-7ae5fe5efcebe601', 'default-9c9df2d6e8ad3e09']

## Models Setup

### Embedding Model

Within the homework you will work with `sentence-transformers/all-MiniLM-L6-v` from fastembed library. <br>
These embedding are precomputed for you in the assignment dataset, but you will need to used model when running the queries.

In [ ]:
## Embeddings are precomputed so you can save some memory by not loading the model
# embedding_model = TextEmbedding('sentence-transformers/all-MiniLM-L6-v2')
embedding_model_size = 384

### Sparse Retrieval Model
Some queries require the prioritization of the certain keywords. <br>
Therefor, you will need to use BM25 algorithm to boost the documents with these keywords during retrieval. <br>
Note that BM25 is not taken into account in the dataset, so you will need to apply when uploading and indexing the data.

In [ ]:
bm25_model = SparseTextEmbedding("Qdrant/bm25")

### Multi-Vector Model
It is general good practice to include reranking model in the IR system. <br>
Reranking uses stronger model to select the most relevant documents from the initial retrieval. <br>
You will implement reranking with multi-vector late interaction embedding ColBERT.

In [ ]:
## Embeddings are precomputed so you can save some memory by not loading the model
# multi_vector_model = LateInteractionTextEmbedding("colbert-ir/colbertv2.0")
multi_vector_model_size = 128

## Database Configuration

### Task 2 - Data Modelling
In this task you will create proper data model for your data including vector representations, index configuration, distance functions and more.

#### Task 2.1 - HNSW Index Configuration
Configure the HNSW index for the retrieval. <br>
**Change the ef_construct parameter to 64 to speed the build time at the cost of the recall.** <br>
We do this for practical reasons, to enable you iterate over the notebook faster.

In [ ]:
# Change ef_construct parameter to 64 to speed the build time at the cost of the recall
ef_construct = 64
hnsw_config=models.HnswConfigDiff(
    ef_construct=ef_construct,
    m=16,
    on_disk=False
)

#### Task 2.2 - Collection Creation
Create model for your data. You should create three vector representations for your data. <br>
There should be one representation for each model defined above. <br>
For multi-vector model make sure to disable the vector index since it will be used only for reranking. <br>
Also, do not forget that multi-vector computation of similarity is not done only through the cosine similarity (check the lecture for more info). <br>
Configure proper modifier for the sparse vector.

In [ ]:
COLLECTION_NAME = "ms_macro"

In [ ]:
try:
    client.delete_collection(COLLECTION_NAME)
    print(f"Deleted existing collection: {COLLECTION_NAME}")
except: 
    print(f"Collection {COLLECTION_NAME} does not exist")

    
collection_created = client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "dense": models.VectorParams(
            hnsw_config=None,
            size=embedding_model_size,
            distance=models.Distance.COSINE
        ),
        "multi_vector": models.VectorParams(
            distance=models.Distance.COSINE,
            size=multi_vector_model_size,
            multivector_config=models.MultiVectorConfig( # Configure max-sim operator
                comparator=models.MultiVectorComparator.MAX_SIM,
            ),
            hnsw_config=models.HnswConfigDiff(m=0), # Disable HNSW index reranking
        )
    },
    sparse_vectors_config={
        "sparse": models.SparseVectorParams(modifier=models.Modifier.IDF)
    }
)

if collection_created:
    print(f"Created collection '{COLLECTION_NAME}'.")
else:
    print("Collection creation failed")


#### Task 2.3 - Create Payload Index & Disable Quantization
Configure keyword payload index for the `groups` field. Make sure that payload index is on-disk.

In [ ]:
payload_index_created = client.create_payload_index(
    collection_name=COLLECTION_NAME,
    field_name="groups",
    field_schema=models.KeywordIndexParams(
        type=models.KeywordIndexType.KEYWORD,
        on_disk=True,
    ),
)

if payload_index_created:
    print(f"Payload index created for field 'groups'")

### Task 3 - Data Upload
Upload vector embeddings and metadata to the created collection, make sure to upload the vectors metadata.

In [ ]:
points: list[models.PointStruct] = []

doc: dict[str, Any]
for doc in documents: # type: ignore
    document_embedding = doc["embedding"]
    bm25_embedding = list(bm25_model.embed([doc["text"]]))[0]
    multi_vector_embedding = doc["multi_vector_embedding"]
    
    point = models.PointStruct(
        id=doc["id"],
        vector={
            "dense": document_embedding,
            "multi_vector": multi_vector_embedding,
            "sparse": models.SparseVector(values=bm25_embedding.values.tolist(), indices=bm25_embedding.indices.tolist()),
        },
        payload={
            "text": doc["text"],
            "groups": doc["groups"]
        }
    )
    points.append(point)

print("Upserting documents...")
client.upload_points(collection_name=COLLECTION_NAME, points=points, batch_size=128)

print(f"Collection info: {client.get_collection(COLLECTION_NAME).points_count} points in collection")
assert client.get_collection(COLLECTION_NAME).points_count == len(documents), f"Expected {len(documents)} points in collection, got {client.get_collection(COLLECTION_NAME).points_count}"

## Querying

### Task 4 - Design Complex Query
Your task is to design a complex query that will include hybrid search, filtering, reranking and metadata boosting. <br>
**The result of this task should be one Qdrant query (do not add any postprocessing logic outside of the Qdrant query)!**
 
**Subtasks:**
1. Define query filter with relation to the `groups` field, do not forget there can be filter values in the query.
    - Think about in which prefetch you should apply the filter.
2. Define sparse and dense search prefetche, the limit for the retrieval should be 100 objects.
3. Define fusion of the two rankings with Reciprocal Rank Fusion (RRF).
4. Rerank the results with ColBERT multi-vector model, use 50 documents for reranking.
5. Boost the results with metadata weighting, use `group_1` with weight 0.05 and `group_2` with weight 0.1.


In [ ]:
def rag_context_retrieval(query: dict[str, Any]) -> QueryResponse:
    query_dense_embedding: list[float] = query["embedding"]
    query_sparse_embedding: SparseEmbedding = list(bm25_model.embed([query["text"]]))[0]
    query_multi_vector_embedding: list[list[float]] = query["multi_vector_embedding"]

    # Task 4.1 - Define query filter
    filter_condition : models.Filter = models.Filter(
        must=[
            models.FieldCondition(
                key="groups",
                match=models.MatchValue(value=filter_value)
            )
            for filter_value in query["filters"]
        ]
    )

    
    sparse_limit = 100
    dense_limit = 100
    # Task 4.2 - Define sparse and dense search. Set their limit to 100.
    prefetch_sparse_and_dense_search: list[models.Prefetch] = [
        models.Prefetch(
            query=models.SparseVector(
                values=query_sparse_embedding.values.tolist(), 
                indices=query_sparse_embedding.indices.tolist()
            ),
            using="sparse",
            limit=sparse_limit,
            filter=filter_condition,
        ),
        models.Prefetch(
            query=query_dense_embedding,
            using="dense",
            limit=dense_limit,
            filter=filter_condition,
        ),
    ]

    # Task 4.3 - Define fusion of the two rankings (set the k parameter of the query to 60 to mitigate effect of high rankings)
    prefetch_fused_rankings: list[models.Prefetch] = [
        models.Prefetch(
            prefetch=prefetch_sparse_and_dense_search,
            query=models.RrfQuery(rrf=models.Rrf(k=60)) # TODO: Modify the fusion query "k"
        )
    ]

    # Task 4.4 - Rerank the results with ColBERT multi-vector model taking 50 documents.
    reranking_limit = 50
    prefetch_multi_vector_reranking: list[models.Prefetch] = [
        models.Prefetch(
            prefetch=prefetch_fused_rankings,
            query=query_multi_vector_embedding,
            using="multi_vector",
            limit=reranking_limit
        )
    ]
    
    group_1_boost_weight = 0.05
    group_2_boost_weight = 0.1
    # Task 4.5 - Boost following "groups" in the search: "group_1" with weight 0.05 and "group_2" with weight 0.1
    reranked_results: QueryResponse = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=prefetch_multi_vector_reranking,
        query=models.FormulaQuery(
            formula=models.SumExpression(sum=[
                "$score",
                models.MultExpression(mult=[group_1_boost_weight, models.FieldCondition(key="groups", match=models.MatchAny(any=["group_1"]))]),
                models.MultExpression(mult=[group_2_boost_weight, models.FieldCondition(key="groups", match=models.MatchAny(any=["group_2"]))])
            ]
        )),
        limit=10
    )

    return reranked_results

In [22]:
avg_retrieval_precision = evaluate_retrieval(rag_context_retrieval, query_dataset)


You achieved 0.9850000000000001 enough to pass ✅!


## Add Results To Query Dataset (Remove For Students!)

In [ ]:
# Remove this for students
store_new_query_result = False
if store_new_query_result:
    print("Evaluating all queries...")
    results_list: list[dict[str, list[Any]]] = []

    train_dataset: Dataset = query_dataset
    for i in range(len(train_dataset)):
        if i % 10 == 0:
            print(f"Processing query {i}/{len(train_dataset)}")
        query_item: dict[str, Any] = cast(dict[str, Any], train_dataset[i])
        result = rag_context_retrieval(query_item)
        # Convert QueryResponse to a serializable format (list of point IDs and scores)
        result_data: dict[str, list[Any]] = {
            "point_ids": [point.id for point in result.points],
            "scores": [point.score for point in result.points]
        }
        results_list.append(result_data)

    # Remove existing 'result' column if it exists
    if "result" in train_dataset.column_names:
        train_dataset = train_dataset.remove_columns("result")
    train_dataset = train_dataset.add_column("result", results_list)
    query_dataset_dict["train"] = train_dataset
    print(f"Added 'result' feature to query_dataset with {len(results_list)} entries")

In [ ]:
# Remove this for students
if store_new_query_result:
    query_dataset_dict.save_to_disk("./data/query-all-MiniLM-L6-v2-100-filters-embedded-results")